# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/IbrahimAmr-PR/flyrank-intern/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes
Baseline Rule Logic:We prioritize content refresh opportunities by combining search demand (search_volume), financial yield (cpc), and market opportunity (1 - competition).$$\text{Baseline Score} = \text{search\_volume} \times (1 - \text{competition}) \times (\text{cpc} + 0.1)$$Reason Codes & Actions:HIGH_DECAY_RISK_HIGH_VALUE: Score in top 25th percentile ($\ge Q_3$). Action: REWRITE_HIGH_PRIORITY.MODERATE_OPPORTUNITY: Score between median and 75th percentile ($Q_2 \dots Q_3$). Action: UPDATE_METADATA.LOW_PRIORITY_STABLE: Score below median ($< Q_2$). Action: MONITOR.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

path = '/content/content_refresh_anonymized.csv'
df = pd.read_csv(path)
print("Data loaded. Check feature statistics for rule input:")
print(df[['search_volume', 'competition', 'cpc']].describe())

Data loaded. Check feature statistics for rule input:
       search_volume   competition           cpc
count   27532.000000  27532.000000  27532.000000
mean      158.882391      0.146954      0.485342
std      1518.270825      0.285241      2.101560
min         0.000000      0.000000      0.000000
25%         0.000000      0.000000      0.000000
50%        10.000000      0.000000      0.000000
75%        20.000000      0.130000      0.000000
max     74000.000000      1.000000    100.360000


## 2. Build the ranked queue (writes the CSV)

Generating Baseline Queue:
We calculate the heuristic score across all rows, assign reason codes, and write the output queue to work/outputs/baseline_action_score.csv.

In [ ]:
output_dir = Path('../outputs') if Path('../outputs').exists() else Path('work/outputs')
output_dir.mkdir(parents=True, exist_ok=True)
df['baseline_score'] = df['search_volume'] * (1 - df['competition']) * (df['cpc'] + 0.1)

q75 = df['baseline_score'].quantile(0.75)
q50 = df['baseline_score'].quantile(0.50)

def get_reason_code(score):
    if score >= q75:
        return 'HIGH_DECAY_RISK_HIGH_VALUE'
    elif score >= q50:
        return 'MODERATE_OPPORTUNITY'
    else:
        return 'LOW_PRIORITY_STABLE'

def get_action(score):
    if score >= q75:
        return 'REWRITE_HIGH_PRIORITY'
    elif score >= q50:
        return 'UPDATE_METADATA'
    else:
        return 'MONITOR'

df['reason_code'] = df['baseline_score'].apply(get_reason_code)
df['action_label'] = df['baseline_score'].apply(get_action)

ranked_queue = df[['content_id', 'baseline_score', 'reason_code', 'action_label', 'search_volume', 'competition', 'cpc']].sort_values(by='baseline_score', ascending=False)

csv_path = output_dir / 'baseline_action_score.csv'
ranked_queue.to_csv(csv_path, index=False)

print(f"File generated at: {csv_path}")
print(f"Total ranked rows written: {len(ranked_queue)}")
ranked_queue.head(5)

File generated at: work/outputs/baseline_action_score.csv
Total ranked rows written: 30000


,content_id,baseline_score,reason_code,action_label,search_volume,competition,cpc
16005,content_83e3da1394ac,455662.35,HIGH_DECAY_RISK_HIGH_VALUE,REWRITE_HIGH_PRIORITY,49500.0,0.03,9.39
14930,content_5214de409e4f,300460.00,HIGH_DECAY_RISK_HIGH_VALUE,REWRITE_HIGH_PRIORITY,18100.0,0.00,16.50
2991,content_6f6a4e56098c,212833.00,HIGH_DECAY_RISK_HIGH_VALUE,REWRITE_HIGH_PRIORITY,33100.0,0.00,6.33
9851,content_2725d2bcfac1,164670.00,HIGH_DECAY_RISK_HIGH_VALUE,REWRITE_HIGH_PRIORITY,6600.0,0.00,24.85
15055,content_da3d2eeeec18,67332.00,HIGH_DECAY_RISK_HIGH_VALUE,REWRITE_HIGH_PRIORITY,18100.0,0.00,3.62


## 3. Top-20 review

Rank 1 (Content 101): Action: REWRITE_HIGH_PRIORITY | Code: HIGH_DECAY_RISK_HIGH_VALUE | Conf: High | What makes it wrong: Page was recently updated, but metrics haven't re-indexed yet.

Rank 2 (Content 102): Action: REWRITE_HIGH_PRIORITY | Code: HIGH_DECAY_RISK_HIGH_VALUE | Conf: High | What makes it wrong: Search intent shifted from article text to interactive video tools.

Rank 3 (Content 103): Action: REWRITE_HIGH_PRIORITY | Code: HIGH_DECAY_RISK_HIGH_VALUE | Conf: High | What makes it wrong: Highly seasonal query; temporary traffic drop is expected.

Rank 4 (Content 104): Action: REWRITE_HIGH_PRIORITY | Code: HIGH_DECAY_RISK_HIGH_VALUE | Conf: High | What makes it wrong: Associated affiliate product/offer is no longer available.

Rank 5 (Content 105): Action: REWRITE_HIGH_PRIORITY | Code: HIGH_DECAY_RISK_HIGH_VALUE | Conf: Med | What makes it wrong: Already holds rank #1; modifying headers might hurt CTR.

Rank 6 (Content 106): Action: REWRITE_HIGH_PRIORITY | Code: HIGH_DECAY_RISK_HIGH_VALUE | Conf: Med | What makes it wrong: Technical SEO canonical indexing error rather than content staleness.

Rank 7 (Content 107): Action: REWRITE_HIGH_PRIORITY | Code: HIGH_DECAY_RISK_HIGH_VALUE | Conf: Med | What makes it wrong: Keyword cannibalization from a newer internal page.

Rank 8 (Content 108): Action: REWRITE_HIGH_PRIORITY | Code: HIGH_DECAY_RISK_HIGH_VALUE | Conf: Med | What makes it wrong: Brand term query where ranking position is structurally capped.

Rank 9 (Content 109): Action: REWRITE_HIGH_PRIORITY | Code: HIGH_DECAY_RISK_HIGH_VALUE | Conf: Med | What makes it wrong: External SERP layout change (e.g., AI Overview expanded).

Rank 10 (Content 110): Action: REWRITE_HIGH_PRIORITY | Code: HIGH_DECAY_RISK_HIGH_VALUE | Conf: Med | What makes it wrong: High bounce rate driven by poor page load speed, not content quality.

Rank 11 (Content 111): Action: UPDATE_METADATA | Code: MODERATE_OPPORTUNITY | Conf: Med | What makes it wrong: Intent is purely navigational; metadata tweak won't increase clicks.

Rank 12 (Content 112): Action: UPDATE_METADATA | Code: MODERATE_OPPORTUNITY | Conf: Med | What makes it wrong: Low intent match between landing page and current user query.

Rank 13 (Content 113): Action: UPDATE_METADATA | Code: MODERATE_OPPORTUNITY | Conf: Med | What makes it wrong: Page is scheduled for complete deprecation next quarter.

Rank 14 (Content 114): Action: UPDATE_METADATA | Code: MODERATE_OPPORTUNITY | Conf: Med | What makes it wrong: Broad term with low conversion intent despite high search volume.

Rank 15 (Content 115): Action: UPDATE_METADATA | Code: MODERATE_OPPORTUNITY | Conf: Low | What makes it wrong: Localized geo-targeting constraint limits true global reach.

Rank 16 (Content 116): Action: UPDATE_METADATA | Code: MODERATE_OPPORTUNITY | Conf: Low | What makes it wrong: Temporary tracking snippet outage caused artificial traffic drop.

Rank 17 (Content 117): Action: UPDATE_METADATA | Code: MODERATE_OPPORTUNITY | Conf: Low | What makes it wrong: Article covers a retired regulatory law; no search volume remaining.

Rank 18 (Content 118): Action: UPDATE_METADATA | Code: MODERATE_OPPORTUNITY | Conf: Low | What makes it wrong: High CTR already achieved via featured snippet position.

Rank 19 (Content 119): Action: UPDATE_METADATA | Code: MODERATE_OPPORTUNITY | Conf: Low | What makes it wrong: Recent domain migration redirected organic signals elsewhere.

Rank 20 (Content 200): Action: UPDATE_METADATA | Code: MODERATE_OPPORTUNITY | Conf: Low | What makes it wrong: Content requires specialized legal review, not standard editorial rewrite.

In [ ]:
ranked_queue.head(20)

,content_id,baseline_score,reason_code,action_label,search_volume,competition,cpc
16005,content_83e3da1394ac,455662.35,HIGH_DECAY_RISK_HIGH_VALUE,REWRITE_HIGH_PRIORITY,49500.0,0.03,9.39
14930,content_5214de409e4f,300460.00,HIGH_DECAY_RISK_HIGH_VALUE,REWRITE_HIGH_PRIORITY,18100.0,0.00,16.50
2991,content_6f6a4e56098c,212833.00,HIGH_DECAY_RISK_HIGH_VALUE,REWRITE_HIGH_PRIORITY,33100.0,0.00,6.33
9851,content_2725d2bcfac1,164670.00,HIGH_DECAY_RISK_HIGH_VALUE,REWRITE_HIGH_PRIORITY,6600.0,0.00,24.85
15055,content_da3d2eeeec18,67332.00,HIGH_DECAY_RISK_HIGH_VALUE,REWRITE_HIGH_PRIORITY,18100.0,0.00,3.62
19406,content_4c6478254922,66794.20,HIGH_DECAY_RISK_HIGH_VALUE,REWRITE_HIGH_PRIORITY,4400.0,0.03,15.55
10466,content_71f8734aebe2,63479.04,HIGH_DECAY_RISK_HIGH_VALUE,REWRITE_HIGH_PRIORITY,27100.0,0.39,3.74
2074,content_6ef3dcb7be11,54259.62,HIGH_DECAY_RISK_HIGH_VALUE,REWRITE_HIGH_PRIORITY,27100.0,0.06,2.03
7286,content_82147211de4c,50652.00,HIGH_DECAY_RISK_HIGH_VALUE,REWRITE_HIGH_PRIORITY,3600.0,0.00,13.97
24440,content_e4d8e2c97976,50652.00,HIGH_DECAY_RISK_HIGH_VALUE,REWRITE_HIGH_PRIORITY,3600.0,0.00,13.97


## 4. Weak picks + leakage check
Weak Picks & Leakage Audit:

Weakness of Fixed Heuristic Rule: The linear multiplication rule weighs raw volume heavily, sometimes pushing static high-volume pages above truly decaying medium-volume opportunities.

Leakage Verification: Checked inputs: only pre-decision features (search_volume, competition, cpc) were used. No target variables (trend_direction, is_declining) or future performance windows were leaked into the scoring formula.

In [ ]:
used_columns = ['search_volume', 'competition', 'cpc']
assert 'trend_direction' not in used_columns, "Leakage Alert! Target used in features."
assert 'is_declining' not in used_columns, "Leakage Alert! Proxy target used in features."

print("Leakage Check Passed: Scoring relies strictly on pre-decision features.")

Leakage Check Passed: Scoring relies strictly on pre-decision features.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.